In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("BancoFutura")
    .master("spark://bf-spark-master:7077")
    .getOrCreate()
)

print(spark)

In [2]:
!pip install loguru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 4.6 MB/s eta 0:00:00


In [3]:
import sys
import os
from pyspark.sql import SparkSession

# ── 1. Path del proyecto ─────────────────────────────
PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

# ── 2. Import del generador ──────────────────────────
from src.ingestion.synthetic_data_generator import BankingDataGenerator

# ── 3. Generar datos (Pandas) ────────────────────────
gen = BankingDataGenerator(n_customers=10000)
df_pd = gen.generate_customers()

# ── 4. Crear SparkSession ────────────────────────────
spark = SparkSession.builder \
    .appName("BancoFutura") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

# ── 5. Convertir a Spark DataFrame ───────────────────
df = spark.createDataFrame(df_pd)

# ── 6. Mostrar datos en Spark ────────────────────────
df.show(5)

2026-05-28 20:36:42.417 | INFO     | src.ingestion.synthetic_data_generator:__init__:68 - Inicializando generador | clientes=10,000 | churn=18% | VIP=8%
2026-05-28 20:36:42.418 | INFO     | src.ingestion.synthetic_data_generator:generate_customers:77 - Generando tabla de clientes...
2026-05-28 20:36:42.450 | SUCCESS  | src.ingestion.synthetic_data_generator:generate_customers:177 - Dataset generado: 10,000 clientes | churn real=20.8% | VIP real=8.2%


   customer_id  age  tenure_months      region        segment  avg_balance_6m  \
0  CLI-0000001   21             26       Maule         Retail        26727.43   
1  CLI-0000002   41            182   Los Lagos  Universitario         4363.88   
2  CLI-0000003   72              1   Los Lagos        Premium         7707.53   
3  CLI-0000004   52             89  Valparaíso  Universitario        26346.34   
4  CLI-0000005   45             59          RM         Retail       593898.41   

   monthly_transactions  app_logins_30d  n_products  clv_score  \
0                    20              10           2   14400.91   
1                     2               1           3    7065.46   
2                    17              14           1   10615.61   
3                    16              11           2   12395.20   
4                    20              13           7   32908.48   

   n_complaints_3m  nps_score  days_since_last_contact last_contact_date  \
0                0         10           